In [7]:
import os, json, base64, mimetypes, time
from pathlib import Path
from openai import OpenAI

In [8]:
from dotenv import load_dotenv
load_dotenv("keys.env") 
client = OpenAI()
DST_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
SRC_DIR = Path("data/rag_chunks")
DST_DIR = Path("data/rag_chunks_image_summary")
MODEL   = "gpt-4.1-mini"   # vision-capable (or gpt-4o if you have access)
PROMPT  = ("You are writing a scientific paper. "
           "Describe the figure precisely: axes/units if visible, key trends, "
           "comparisons, anomalies, and the main takeaway. Be specific.")


In [10]:
def _data_url(img_path: Path) -> str:
    mime = mimetypes.guess_type(str(img_path))[0] or "image/jpeg"
    b64  = base64.b64encode(img_path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def summarize_image(img_path: Path) -> str:
    url = _data_url(img_path)
    resp = client.responses.create(
        model=MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": PROMPT},
                {"type": "input_image", "image_url": url},
            ],
        }],
        stream=False,
    )
    return resp.output_text.strip()

In [11]:
def process_file(src_json: Path, root: Path = Path(".")):
    data = json.loads(src_json.read_text())
    # support either list of chunks or {"chunks":[...]}
    chunks = data.get("chunks") if isinstance(data, dict) else data
    if not isinstance(chunks, list):
        raise ValueError(f"Unexpected JSON structure in {src_json}")

    for ch in chunks:
        try:
            if ch.get("type") == "figure":
                md = ch.setdefault("metadata", {})
                img_rel = md.get("image_path") or md.get("image")  # be tolerant
                if img_rel:
                    img_path = (root / img_rel).resolve()
                    if img_path.exists():
                        # simple retry for transient errors
                        for attempt in range(4):
                            try:
                                md["image_summary"] = summarize_image(img_path)
                                break
                            except Exception as e:
                                if attempt == 3: 
                                    md["image_summary"] = f"[error] {type(e).__name__}: {e}"
                                time.sleep(1.5 * (2 ** attempt))
                    else:
                        md["image_summary"] = f"[missing image at {img_rel}]"
        except Exception as e:
            ch.setdefault("metadata", {})["image_summary"] = f"[error] {type(e).__name__}: {e}"

    # write out preserving original shape
    out = {"chunks": chunks} if isinstance(data, dict) and "chunks" in data else chunks
    (DST_DIR / src_json.name).write_text(json.dumps(out, ensure_ascii=False, indent=2))


In [12]:
for f in sorted(SRC_DIR.glob("*.json")):
    process_file(f, root=Path("."))  

print("Done ➜", DST_DIR)

Done ➜ data/rag_chunks_image_summary


In [ ]:
import os
print("CWD:", os.getcwd())
print("SRC exists:", SRC_DIR.exists(), "DST exists:", DST_DIR.exists())
